# Exp 1 expanded dataset: generalization with validation early stopping
Train separate ACS 2023 and ACS 2024 income MLPs. Train a separate membership
attack from each auxiliary year, ACS 2014 and ACS 2018, using 16 shadows per year
(4 random forests, 4 gradient boosting, 4 XGBoost, 4 generic MLPs). Report all four combinations:
2023/2014, 2023/2018, 2024/2014 and 2024/2018 (target/auxiliary).

Use the explicitly approved common 47-feature schema: VPS is absent from the
2024 file and is excluded from all four years. Other feature definitions,
nominal income >$50,000 labels, eligibility, training settings and split sizes
follow Exp 3. The historical 48-input metrics are a separate configuration.
Every target and shadow fits its own preprocessor on training records only.

Blind walks are enabled, with the same numeric protocol for shadows and targets.
All 16 shadows for one auxiliary year share that year's grouped attack split;
auxiliary validation alone selects thresholds. Target membership labels only
score the frozen attacks. Native category codes are retained within each year.

Extract target attack features once per target and reuse them for both auxiliary
attacks. Train and query one shadow at a time to keep memory use bounded. No
model checkpoints or log files are loaded or saved. Compact progress, metrics,
row assignments and the study protocol are exported. The known sklearn/joblib
warning is suppressed; other warnings and exceptions remain visible.

This notebook stops after complete-feature baseline MIA. PFV, missing-feature
reconstruction and uncertainty-aware methods are not run here. Notebook creation
and validation execute no experiments; ask before running the cells.

Generalization variant of `exp_1_expanded dataset.ipynb`: the only target-training
changes are enabling validation-loss early stopping and restoring the best epoch.
Both target MLPs keep a maximum of 100 epochs and stop after five consecutive
epochs without a strictly lower validation loss. The existing patience of five
is made explicit. Validation is separate from test and MIA nonmember records;
test accuracy, membership scores and the train/test gap never select epochs.
Architecture, learning rate, dropout, weight decay, features, splits, seeds,
tree-shadow settings, attack threshold selection and blind walks follow the source notebook.
Early stopping aims to reduce overfitting; higher accuracy or a smaller gap is
not guaranteed. New metrics and row assignments go to a separate directory.

Added generic MLP shadows use predeclared hidden layers 128/64, no dropout,
Adam learning rate 0.001 and weight decay 0.0001. They select epochs on their own
shadow validation loss, with patience five, a 50-epoch maximum and best weights
restored. Nonmember records and target results never select MLP epochs. There
are still 16 shadows per auxiliary year, four per algorithm. The final membership
classifier remains RF. Target training and feature extraction are unchanged.
This mixed-shadow variant writes to its own directory; old artifacts are retained.

Additional end sections test matched MLP-only shadows from unused ACS 2023 and
2024 records against the respective same-year target only. They add rich-output
RF attacks, subgroup and top-5% outlier diagnostics, generalization-loss gaps,
and offline LiRA. Existing 2014/2018 mixed-shadow comparisons remain unchanged.


In [2]:
from pathlib import Path
import csv
import gc
import importlib
import json
import time
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from src.data import split_acs_data, subset_target_training
import src.expanded_data as expanded_data_module
# Refresh the shared helper when an existing kernel cached the old notebook filename.
importlib.reload(expanded_data_module)
from src.expanded_data import reference_split_config
import src.shadow_models as shadow_models_module
importlib.reload(shadow_models_module)
import src.multiyear_baseline as multiyear_baseline_module
importlib.reload(multiyear_baseline_module)
from src.multiyear_baseline import (
    BASELINE_FEATURES, baseline_raw_path, load_baseline_year,
    preprocess_baseline_partitions, baseline_feature_inventory, train_baseline_shadow,
)
from src.training import train_target_model, evaluate_model
from src.shadow_models import PAPER_SHADOW_ALGORITHMS, MLP_SHADOW_CONFIG, summarize_shadow_models
from src.membership_attack import train_membership_classifier, evaluate_attack
from src.paper_attack_features import (
    collect_shadow_paper_features, collect_target_paper_features, make_probability_oracle,
)
import src.notebook_output as notebook_output_module
importlib.reload(notebook_output_module)
notebook_output_module.suppress_sklearn_parallel_warning()
from src.notebook_output import CellProgress, quiet_output

torch.set_num_threads(4)
SEED = 42
TARGET_YEARS = (2023, 2024)
AUXILIARY_YEARS = (2014, 2018)
TRAIN_SIZE = 100_000
N_SHADOW_MODELS = 16
SHADOW_RECORDS_PER_CLASS = 2000
SHADOW_ALGORITHMS = PAPER_SHADOW_ALGORITHMS + ('mlp',)
SHADOW_MLP_CONFIG = dict(MLP_SHADOW_CONFIG)
assert N_SHADOW_MODELS == 4 * len(SHADOW_ALGORITHMS)
FEATURE_COLUMNS = list(BASELINE_FEATURES)
assert len(FEATURE_COLUMNS) == 47 and 'VPS' not in FEATURE_COLUMNS
DATA_ROOT = Path('data/raw/folktables')
SPLIT_CONFIG = reference_split_config()
assert SPLIT_CONFIG['seed'] == SEED
TRAIN_CONFIG = dict(hidden_dims=(512, 256, 128), dropout=0.0, epochs=100,
    batch_size=256, learning_rate=1e-3, weight_decay=0.0, seed=SEED,
    device='cpu', early_stopping=True, restore_best=True, monitor_train=True, patience=5)
WALK_CONFIG = dict(max_steps=20, repeats=2, categorical_change_probability=0.05,
    batch_size=512, seed=SEED)
EXPANDED_NUMERIC_WALK = {'JWMNP': (2.0, 0, 200), 'JWRIP': (1.0, 1, 10)}
INCLUDE_BLIND_WALK = True
ATTACK_FEATURE_SET = 'paper' if INCLUDE_BLIND_WALK else 'paper_no_walk'
ATTACK_FEATURES = (['class_scaled_logit', 'blind_walk', 'entropy', 'cross_entropy']
    if INCLUDE_BLIND_WALK else ['class_scaled_logit', 'entropy', 'cross_entropy'])
ATTACK_MODE = 'with_walk_consistent_v1' if INCLUDE_BLIND_WALK else 'probability_only_v1'
RUN_DIR = Path('artifacts/exp_1_generalization_targets2023_2024_aux2014_2018_s16_common47_mixed_mlp') / ATTACK_MODE
RUN_DIR.mkdir(parents=True, exist_ok=True)
print('Targets:', TARGET_YEARS, '| Auxiliary pools:', AUXILIARY_YEARS)
print('Inputs:', len(FEATURE_COLUMNS), '| Shadows per auxiliary year:', N_SHADOW_MODELS)
print('Attack features:', ATTACK_FEATURES)
print('Results:', RUN_DIR)


Targets: (2023, 2024) | Auxiliary pools: (2014, 2018)
Inputs: 47 | Shadows per auxiliary year: 16
Attack features: ['class_scaled_logit', 'blind_walk', 'entropy', 'cross_entropy']
Results: artifacts\exp_1_generalization_targets2023_2024_aux2014_2018_s16_common47_mixed_mlp\with_walk_consistent_v1


## Check all source files before training
Check headers for all four years first. Income and survey weights are used only
for eligibility/labels. They are excluded from the 47 predictors. Missing source
columns cause an early error rather than silent imputation of unavailable inputs.


In [3]:
source_checks = []
for year in TARGET_YEARS + AUXILIARY_YEARS:
    path = baseline_raw_path(year, DATA_ROOT)
    if not path.is_file():
        raise FileNotFoundError(f'ACS {year} source CSV is required: {path}')
    with path.open(encoding='utf-8-sig', newline='') as handle:
        columns = set(next(csv.reader(handle, skipinitialspace=True)))
    missing = set(FEATURE_COLUMNS + ['PINCP', 'PWGTP']) - columns
    if missing:
        raise ValueError(f'ACS {year} missing required columns: {sorted(missing)}')
    source_checks.append(dict(year=year, role='target' if year in TARGET_YEARS else 'auxiliary',
        file=str(path), predictors=len(FEATURE_COLUMNS)))
display(pd.DataFrame(source_checks))


,year,role,file,predictors
0,2023,target,data\raw\folktables\2023\1-Year\psam_p06.csv,47
1,2024,target,data\raw\folktables\2024\1-Year\psam_p06.csv,47
2,2014,auxiliary,data\raw\folktables\2014\1-Year\ss14pca.csv,47
3,2018,auxiliary,data\raw\folktables\2018\1-Year\psam_p06.csv,47


## Train and evaluate two independent target MLPs
Each target uses 100,000 training records and its own stratified validation,
income-test and MIA partitions. MIA members belong to training; MIA nonmembers
are held out. Each year has 10,000 MIA members and 10,000 MIA nonmembers.
Targets train for up to 100 epochs, stopping after five consecutive epochs
without a strictly lower validation loss. Restore the lowest-validation-loss
weights before income and MIA evaluation. The same fixed split configuration
is applied independently within each year.


In [4]:
def prepare_target(target_year, progress):
    started = time.monotonic()
    progress.update(f'Target {target_year}: loading and splitting')
    source = load_baseline_year(target_year, data_root=DATA_ROOT)
    reference = split_acs_data(source, **SPLIT_CONFIG)
    partitions = subset_target_training(reference, train_size=TRAIN_SIZE, seed=SEED)
    for name in ['validation', 'test', 'mia_members', 'mia_nonmembers']:
        np.testing.assert_array_equal(partitions[name].index, reference[name].index)
    train_ids = set(partitions['train'].index)
    assert set(partitions['mia_members'].index).issubset(train_ids)
    for name in ['validation', 'test', 'mia_nonmembers']:
        assert train_ids.isdisjoint(partitions[name].index)
    progress.update(f'Target {target_year}: preprocessing and training up to {TRAIN_CONFIG["epochs"]} epochs (validation patience=5)')
    with quiet_output():
        preprocessor, arrays = preprocess_baseline_partitions(partitions)
        model, history, training = train_target_model(
            arrays['train']['X'], arrays['train']['y'],
            arrays['validation']['X'], arrays['validation']['y'], **TRAIN_CONFIG)
        income_metrics = {name: evaluate_model(model, arrays[name]['X'], arrays[name]['y'])
            for name in ['train', 'validation', 'test']}
    assert list(preprocessor.feature_names_in_) == FEATURE_COLUMNS
    progress.update(f'Target {target_year}: extracting baseline attack features once')
    with quiet_output():
        attack_features, query_audit = collect_target_paper_features(
            model, preprocessor, partitions, config=WALK_CONFIG, seed=SEED,
            numeric_protocol=EXPANDED_NUMERIC_WALK, include_blind_walk=INCLUDE_BLIND_WALK)
    assert list(attack_features[ATTACK_FEATURES].columns) == ATTACK_FEATURES
    directory = RUN_DIR / f'target_{target_year}'
    directory.mkdir(parents=True, exist_ok=True)
    for name, frame in partitions.items():
        pd.Series(frame.index, name='record_id').to_csv(directory / f'{name}_ids.csv', index=False)
    baseline_feature_inventory(partitions['train']).to_csv(directory / 'feature_inventory.csv')
    pd.DataFrame(income_metrics).T.to_csv(directory / 'income_metrics.csv')
    progress.update(f'Target {target_year}: complete ({time.monotonic() - started:.1f}s)')
    return dict(model=model, preprocessor=preprocessor, partitions=partitions,
        attack_features=attack_features, income_metrics=income_metrics, history=history,
        training=training, query_audit=query_audit, eligible_records=len(source),
        income_adjustment=source.attrs['income_adjustment'])


In [5]:
target_results = {}
target_progress = CellProgress()
for target_year in TARGET_YEARS:
    target_results[target_year] = prepare_target(target_year, target_progress)
    gc.collect()
target_income_rows = [dict(target_year=year, partition=name, **metrics)
    for year, result in target_results.items()
    for name, metrics in result['income_metrics'].items()]
target_income_metrics = pd.DataFrame(target_income_rows)
display(target_income_metrics.set_index(['target_year', 'partition']).round(4))
print('Two target models are available in target_results[2023] and target_results[2024].')


accuracy  roc_auc  log_loss
target_year partition                              
2023        train         0.8438   0.9243    0.3521
            validation    0.8235   0.9042    0.3928
            test          0.8253   0.9076    0.3850
2024        train         0.8407   0.9233    0.3550
            validation    0.8203   0.9040    0.3907
            test          0.8188   0.9007    0.3974

Two target models are available in target_results[2023] and target_results[2024].


## Fit separate baseline attacks from ACS 2014 and ACS 2018
For each auxiliary year, train 16 shadows independently: cycle RF, gradient
boosting, XGBoost and generic MLP with seeds 42, 52, ..., 192. This gives four
models per algorithm. Preserve the original 40/20/40 shadow splits and tree
estimator settings. Generic MLP settings are recorded in SHADOW_MLP_CONFIG;
shadow validation loss alone selects their epochs. Each shadow
contributes 2,000 sampled members and 2,000 sampled nonmembers to the attack.

Query one shadow at a time and discard its fitted model after feature extraction.
The resulting features match collecting the same ordered models as one pool.
Record-ID-grouped 60/20/20 attack splits keep repeated records in one role. Fit
one RF attack per auxiliary year and select its thresholds on that year's
validation records. The two auxiliary pools are not combined or selected using
target scores. Both targets use each auxiliary attack's frozen thresholds.


In [6]:
def fit_query_one_shadow(auxiliary, name, algorithm, seed):
    with quiet_output():
        shadow = train_baseline_shadow(auxiliary, algorithm, seed=seed,
            parameters=SHADOW_MLP_CONFIG if algorithm == 'mlp' else None)
        shadow['attack_data']['shadow_model'] = name
        audit_rows = shadow['attack_data'].iloc[:32]
        oracle = make_probability_oracle(shadow['model'], shadow['preprocessor'], schema='acs')
        native = auxiliary.loc[audit_rows['record_id'], FEATURE_COLUMNS]
        np.testing.assert_allclose(oracle(native),
            audit_rows[['p_income_le_50k', 'p_income_gt_50k']].to_numpy(),
            rtol=1e-5, atol=1e-6, err_msg=f'Query mismatch: {name}')
        features, audits = collect_shadow_paper_features({name: shadow}, auxiliary,
            per_class=SHADOW_RECORDS_PER_CLASS, config=WALK_CONFIG, seed=SEED,
            numeric_protocol=EXPANDED_NUMERIC_WALK, include_blind_walk=INCLUDE_BLIND_WALK)
        metrics = summarize_shadow_models({name: shadow})
    return features, audits[name], metrics


def prepare_auxiliary_attack(auxiliary_year, progress):
    started = time.monotonic()
    auxiliary = load_baseline_year(auxiliary_year, data_root=DATA_ROOT)
    for target_year, target in target_results.items():
        for frame in target['partitions'].values():
            assert set(auxiliary.index).isdisjoint(frame.index)
    feature_tables, shadow_metric_tables, query_audits, shadow_seeds = [], [], {}, {}
    for index in range(N_SHADOW_MODELS):
        algorithm = SHADOW_ALGORITHMS[index % len(SHADOW_ALGORITHMS)]
        name = f'{algorithm}_{index + 1:02d}'
        seed = SEED + 10 * index
        progress.update(f'Auxiliary {auxiliary_year}: shadow [{index + 1}/{N_SHADOW_MODELS}] {name}')
        features, audit, metrics = fit_query_one_shadow(auxiliary, name, algorithm, seed)
        feature_tables.append(features)
        shadow_metric_tables.append(metrics)
        query_audits[name] = audit
        shadow_seeds[name] = seed
        gc.collect()
    attack_data = pd.concat(feature_tables, ignore_index=True)
    assert len(attack_data) == N_SHADOW_MODELS * 2 * SHADOW_RECORDS_PER_CLASS
    progress.update(f'Auxiliary {auxiliary_year}: fitting attack and calibrating thresholds')
    with quiet_output():
        attack = train_membership_classifier(attack_data, seed=SEED,
            algorithm='random_forest', feature_set=ATTACK_FEATURE_SET)
    assert list(attack['classifier'].feature_names_in_) == ATTACK_FEATURES
    role_ids = {name: set(frame['record_id']) for name, frame in attack['splits'].items()}
    assert role_ids['train'].isdisjoint(role_ids['validation'])
    assert role_ids['train'].isdisjoint(role_ids['test'])
    assert role_ids['validation'].isdisjoint(role_ids['test'])
    directory = RUN_DIR / f'auxiliary_{auxiliary_year}'
    directory.mkdir(parents=True, exist_ok=True)
    pd.concat(shadow_metric_tables).to_csv(directory / 'shadow_income_metrics.csv')
    pd.DataFrame(attack['metrics']).T.to_csv(directory / 'attack_metrics.csv')
    for name, frame in attack['splits'].items():
        frame[['record_id', 'shadow_model', 'membership']].to_csv(
            directory / f'attack_{name}_rows.csv', index=False)
    progress.update(f'Auxiliary {auxiliary_year}: complete ({time.monotonic() - started:.1f}s)')
    return dict(attack_result=attack, query_audits=query_audits, shadow_seeds=shadow_seeds,
        eligible_records=len(auxiliary), income_adjustment=auxiliary.attrs['income_adjustment'])


In [7]:
auxiliary_results = {}
auxiliary_progress = CellProgress()
for auxiliary_year in AUXILIARY_YEARS:
    auxiliary_results[auxiliary_year] = prepare_auxiliary_attack(auxiliary_year, auxiliary_progress)
    gc.collect()
auxiliary_attack_rows = [dict(auxiliary_year=year, partition=name, **metrics)
    for year, result in auxiliary_results.items()
    for name, metrics in result['attack_result']['metrics'].items()]
auxiliary_attack_metrics = pd.DataFrame(auxiliary_attack_rows)
display(auxiliary_attack_metrics.set_index(['auxiliary_year', 'partition'])[
    ['accuracy', 'balanced_accuracy', 'roc_auc', 'tpr_at_1pct_fpr_roc']].round(4))


accuracy  balanced_accuracy  roc_auc  \
auxiliary_year partition                                          
2014           train         0.5974             0.5968   0.6428   
               validation    0.5205             0.5213   0.5260   
               test          0.4998             0.5009   0.5019   
2018           train         0.6061             0.6061   0.6481   
               validation    0.5179             0.5185   0.5206   
               test          0.5002             0.4997   0.5045   

                           tpr_at_1pct_fpr_roc  
auxiliary_year partition                        
2014           train                    0.0359  
               validation               0.0111  
               test                     0.0078  
2018           train                    0.0378  
               validation               0.0107  
               test                     0.0101

## Baseline MIA: evaluate all four target/auxiliary pairs
Target features were extracted once per target, independently of auxiliary
membership labels. Reuse them here; changing auxiliary year adds no target-oracle
queries. Every target uses identical MIA records for its two auxiliary comparisons.

Accuracy and balanced accuracy use the auxiliary-validation-selected cutoff.
ROC-derived TPR@1% FPR is an evaluation summary. Separate TPR/FPR columns report
the actual operating point of the frozen auxiliary low-FPR cutoff; its target FPR
need not equal 1%. No target labels choose cutoffs or the auxiliary year.


In [8]:
baseline_rows = []
for target_year in TARGET_YEARS:
    features = target_results[target_year]['attack_features']
    for auxiliary_year in AUXILIARY_YEARS:
        attack = auxiliary_results[auxiliary_year]['attack_result']
        with quiet_output():
            metrics, scores = evaluate_attack(attack['classifier'], features,
                threshold=attack['accuracy_threshold'], low_fpr_threshold=attack['low_fpr_threshold'])
        metrics['tpr_at_accuracy_threshold'] = metrics['tp'] / metrics['members']
        metrics['fpr_at_accuracy_threshold'] = metrics['fp'] / (metrics['records'] - metrics['members'])
        baseline_rows.append(dict(target_year=target_year, auxiliary_year=auxiliary_year,
            n_shadow_models=N_SHADOW_MODELS, **metrics))
        features.assign(membership_score=scores).to_csv(
            RUN_DIR / f'target_{target_year}_auxiliary_{auxiliary_year}_scores.csv', index=False)
baseline_mia_metrics = pd.DataFrame(baseline_rows).set_index(['target_year', 'auxiliary_year'])
assert len(baseline_mia_metrics) == len(TARGET_YEARS) * len(AUXILIARY_YEARS)
display(baseline_mia_metrics[['accuracy', 'balanced_accuracy', 'roc_auc',
    'tpr_at_1pct_fpr_roc', 'tpr_at_shadow_threshold', 'fpr_at_shadow_threshold']].round(4))


accuracy  balanced_accuracy  roc_auc  \
target_year auxiliary_year                                         
2023        2014              0.5073             0.5073   0.5082   
            2018              0.5044             0.5044   0.5086   
2024        2014              0.5058             0.5058   0.5087   
            2018              0.5104             0.5104   0.5099   

                            tpr_at_1pct_fpr_roc  tpr_at_shadow_threshold  \
target_year auxiliary_year                                                 
2023        2014                         0.0113                   0.0139   
            2018                         0.0112                   0.0129   
2024        2014                         0.0107                   0.0117   
            2018                         0.0118                   0.0100   

                            fpr_at_shadow_threshold  
target_year auxiliary_year                           
2023        2014                             0.0124  
            2018                             0.0114  
2024        2014                             0.0108  
            2018                             0.0089

## Save final metrics and protocol
CSV/JSON outputs contain metrics, predictions and row assignments, not fitted
models or resumable checkpoints. Each year's attack result remains in memory for
later inspection; there is no missing-feature or uncertainty computation here.


In [9]:
baseline_mia_metrics.to_csv(RUN_DIR / 'baseline_mia_metrics.csv')
target_income_metrics.to_csv(RUN_DIR / 'target_income_metrics.csv', index=False)
auxiliary_attack_metrics.to_csv(RUN_DIR / 'auxiliary_attack_metrics.csv', index=False)
protocol = dict(execution_status='baseline_mia_completed',
    target_years=TARGET_YEARS, auxiliary_years=AUXILIARY_YEARS,
    pairs=[dict(target_year=t, auxiliary_year=a) for t in TARGET_YEARS for a in AUXILIARY_YEARS],
    feature_count=len(FEATURE_COLUMNS), features=FEATURE_COLUMNS,
    schema_change='VPS excluded from every year with user approval; absent from 2024 file',
    label='nominal PINCP > 50000', eligibility='AGEP > 16; PINCP > 100; WKHP > 0; PWGTP >= 1',
    train_size=TRAIN_SIZE, split_config=SPLIT_CONFIG, target_training_config=TRAIN_CONFIG,
    shadow_models_per_auxiliary=N_SHADOW_MODELS, total_shadow_models=N_SHADOW_MODELS * len(AUXILIARY_YEARS),
    shadow_algorithms=list(SHADOW_ALGORITHMS), records_per_class_per_shadow=SHADOW_RECORDS_PER_CLASS,
    shadow_models_per_algorithm=4, shadow_mlp_config=SHADOW_MLP_CONFIG,
    attack_feature_set=ATTACK_FEATURE_SET, attack_features=ATTACK_FEATURES,
    include_blind_walk=INCLUDE_BLIND_WALK, walk_config=WALK_CONFIG,
    extra_numeric_protocol=EXPANDED_NUMERIC_WALK,
    walk_protocol_scope='Same numeric fields and perturbation settings for all target and shadow years',
    threshold_selection='Each auxiliary validation only; frozen for both targets',
    target_features='Queried once per target and reused for both auxiliary attacks',
    checkpoints_enabled=False, logging_enabled=False,
    targets={str(year): dict(training=result['training'], eligible_records=result['eligible_records'],
        income_metrics=result['income_metrics'], query_audit=result['query_audit'])
        for year, result in target_results.items()},
    auxiliary={str(year): dict(eligible_records=result['eligible_records'],
        shadow_seeds=result['shadow_seeds'], attack_metrics=result['attack_result']['metrics'],
        query_audits=result['query_audits']) for year, result in auxiliary_results.items()},
    limitations=['native category codes differ across years', 'one training/attack seed configuration',
        '47-feature schema differs from historical 48-feature baseline'])
_ = (RUN_DIR / 'protocol.json').write_text(json.dumps(protocol, indent=2), encoding='utf-8')
print('Complete-feature baseline MIA finished. Results:', RUN_DIR / 'baseline_mia_metrics.csv')


Complete-feature baseline MIA finished. Results: artifacts\exp_1_generalization_targets2023_2024_aux2014_2018_s16_common47_mixed_mlp\with_walk_consistent_v1\baseline_mia_metrics.csv


The cross-year baseline ends here. The separate same-year sections below add
matched MLP shadow attacks and diagnostics; no PFV or reconstruction is added.


## Same-year matched MLP shadows: 2023→2023 and 2024→2024
Use **16 additional MLP shadows per year**. Match each target's MLP hidden layers,
optimizer, learning rate, dropout, weight decay, batch size, maximum epochs and
validation-loss stopping/restoration rule. Reference split/initialization seeds
remain independent (42, 52, ..., 192). Fit the same 47-feature preprocessing
recipe independently on each shadow's training rows. Fitted statistics,
vocabularies and hence encoded input dimensions need not equal the target's;
no target-fitted preprocessing or weights are given to shadows.

The source for each pool is that year's eligible rows **unused by ALL target
partitions**, including validation, income test and MIA nonmembers. Target
training remains 100,000 records with all its current IDs unchanged. The new
75/10/15 train/validation/test split is used when early stopping is enabled.
Without early stopping, use 80/20 train/test and disable best-weight restoration;
no validation or test data enters the training helper in that fixed-schedule mode.
Each model independently randomizes its stratified split with its own seed.
Approximately 73,000 unused records remain per year, giving about 55,000 training
rows per shadow with 75/10/15, so **training sample size is still not matched**.
Counts and zero-overlap assertions are reported; records are row
IDs, not deduplicated people. References can overlap each other, as before.

The new RF attack sees both output probabilities, true-class probability,
confidence, entropy, stable cross entropy, correctness, true-class logit margin,
modified entropy and the existing blind-walk feature. This is a predeclared
richer-feature comparison, not an isolated test of auxiliary year alone.
Assign record-group attack roles before sampling predictions. Retain every
shadow OUT-test prediction; independently subsample shadow IN-training rows to
match that exact OUT count within each attack role. The standard meta-classifier's
training, validation and test datasets are therefore each exactly 50/50 IN/OUT.
Shadow validation rows are never attack IN or OUT. Using all OUT rows increases
blind-walk query work compared with the earlier 2,000-per-class cap.
Repeated IDs keep one grouped attack train/validation/test role. No target MIA
labels choose settings, score direction, thresholds, groups or the anomaly model.

Creating these cells does not authorize execution. Running this section trains
32 additional non-private MLPs (in addition to the 32 existing cross-year
shadows). No DP notebook is modified and no target is retrained here.


In [10]:
import src.training as same_year_training_module
importlib.reload(same_year_training_module)
import src.multiyear_baseline as same_year_baseline_module
importlib.reload(same_year_baseline_module)
import src.same_year_mia as same_year_mia_module
importlib.reload(same_year_mia_module)
from src.same_year_mia import (
    disjoint_same_year_pool, verify_target_recipe, run_same_year_study,
    fit_outlier_slices, score_metrics, subgroup_metrics,
    fit_offline_calibration, gaussian_out_score,
)

SAME_YEAR_REFERENCE_COUNT = N_SHADOW_MODELS  # 16, matching the existing pool count
SAME_YEAR_MINIMUM_OUT = 3
SAME_YEAR_TARGETS = (2023, 2024)
SAME_YEAR_SPLIT_POLICY = ('train75_val10_test15' if TRAIN_CONFIG['early_stopping'] else 'train80_test20')
SAME_YEAR_DIR = RUN_DIR / f'same_year_matched_mlp_{SAME_YEAR_SPLIT_POLICY}'
assert SAME_YEAR_REFERENCE_COUNT == 16
assert set(SAME_YEAR_TARGETS) <= set(target_results)
same_year_sources, same_year_audit_parts = {}, []
for year in SAME_YEAR_TARGETS:
    verify_target_recipe(target_results[year], TRAIN_CONFIG)
    same_year_sources[year] = load_baseline_year(year, data_root=DATA_ROOT)
    unused, audit = disjoint_same_year_pool(same_year_sources[year], target_results[year]['partitions'])
    same_year_audit_parts.append(audit.assign(target_year=year))
    del unused
same_year_disjointness = pd.concat(same_year_audit_parts, ignore_index=True)
display(same_year_disjointness.set_index(['target_year', 'target_partition']))
print('Additional matched MLP shadows:', SAME_YEAR_REFERENCE_COUNT * len(SAME_YEAR_TARGETS))


target_records  shadow_pool_records  \
target_year target_partition                                        
2023        validation                 10000                73278   
            test                       10000                73278   
            mia_nonmembers             10000                73278   
            train                     100000                73278   
            mia_members                10000                73278   
2024        validation                 10000                73591   
            test                       10000                73591   
            mia_nonmembers             10000                73591   
            train                     100000                73591   
            mia_members                10000                73591   

                              overlapping_record_ids  
target_year target_partition                          
2023        validation                             0  
            test                                   0  
            mia_nonmembers                         0  
            train                                  0  
            mia_members                            0  
2024        validation                             0  
            test                                   0  
            mia_nonmembers                         0  
            train                                  0  
            mia_members                            0

Additional matched MLP shadows: 32


### Run the two same-year shadow pools and rich-output RF attacks
This is an experiment: execute only after explicit approval. Each pool is used
only against its respective target. The targets and their original MIA record
sets are reused. Keep blind-walk settings unchanged; query each target's existing
walk feature from memory rather than running its walks again. The references
also query all shadow-pool and target-candidate unaugmented logits once, so the
offline LiRA section reuses scores instead of fitting a second reference ensemble.
Training models and reference scores stay in memory; no checkpoints are used.


In [11]:
same_year_results = {}
same_year_progress = CellProgress()
for year in SAME_YEAR_TARGETS:
    same_year_results[year] = run_same_year_study(
        same_year_sources[year], target_results[year], target_year=year,
        count=SAME_YEAR_REFERENCE_COUNT, train_config=TRAIN_CONFIG,
        walk_config=WALK_CONFIG, numeric_protocol=EXPANDED_NUMERIC_WALK,
        include_blind_walk=INCLUDE_BLIND_WALK,
        seed=SEED, progress=same_year_progress.update)
    gc.collect()
same_year_progress.update('Both same-year matched shadow pools and rich RF attacks completed')


## Same-year RF metrics, subgroup/outlier breakdown and loss gaps
Fit a fixed Isolation Forest (100 trees, max_samples=256, seed 42) using only the
disjoint same-year shadow pool and its own training-fitted preprocessing. Rank
target candidates without membership labels and select exactly the top 5%
(1,000 of 20,000) by anomaly score, with stable tie handling. This is an input
anomaly diagnostic, not proof of statistical leverage or rare-group coverage.

Report whole-set, top-5%, remaining-95%, income class, SEX and RAC1P slices for
both attacks. Use the same global shadow-validation-selected cutoffs in every
slice. Report membership counts/proportions, majority-class baseline, balanced
accuracy, AUC, TPR@1%FPR and actual transferred-cutoff FPR/TPR alongside accuracy.
Slices need not have 50% members: raw accuracy alone can be misleading. AUC and
balanced accuracy are undefined for a single-membership-class slice; keep NaN
with an explicit status. Small slices have coarse low-FPR resolution.

Report target and shadow train-versus-validation loss/accuracy gaps. The signed
loss gap is **validation loss minus training loss** (positive means validation
loss is worse), not training minus validation. Accuracy confidence intervals are
descriptive Wilson intervals; they do not prove equivalence to chance or account
for all training randomness and multiple subgroup comparisons.
In 80/20 mode there is no shadow validation split, so its validation-gap columns
are undefined; final shadow test gaps are still reported after training.


In [12]:
same_year_metric_rows, same_year_aux_rows, same_year_slice_parts = [], [], []
same_year_gap_rows, same_year_shadow_gap_parts = [], []
for year, result in same_year_results.items():
    same_year_progress.update(f'{year}: fitting input-only outlier diagnostic and scoring RF slices')
    anomaly_scores, outlier_mask = fit_outlier_slices(result['pool'], result['candidates'], seed=SEED)
    result['target_rows']['anomaly_score'] = anomaly_scores
    result['target_rows']['outlier_top5'] = outlier_mask
    attack = result['rich_attack']
    row = score_metrics(result['target_rows'].membership, result['target_rows'].rich_rf_score,
        threshold=attack['accuracy_threshold'], low_fpr_threshold=attack['low_fpr_threshold'])
    same_year_metric_rows.append(dict(target_year=year, auxiliary_year=year, method='matched_rich_rf', **row))
    for role, metrics in attack['metrics'].items():
        same_year_aux_rows.append(dict(auxiliary_year=year, method='matched_rich_rf', partition=role, **metrics))
    same_year_slice_parts.append(subgroup_metrics(result['target_rows'], result['candidates'], outlier_mask,
        score_column='rich_rf_score', threshold=attack['accuracy_threshold'],
        low_fpr_threshold=attack['low_fpr_threshold']).assign(target_year=year, method='matched_rich_rf'))
    same_year_gap_rows.append(dict(target_year=year, **result['target_generalization']))
    same_year_shadow_gap_parts.append(result['shadow_generalization'].assign(auxiliary_year=year))

same_year_target_generalization = pd.DataFrame(same_year_gap_rows)
same_year_shadow_generalization = pd.concat(same_year_shadow_gap_parts, ignore_index=True)
display(pd.DataFrame(same_year_metric_rows).set_index(['target_year', 'method'])[
    ['accuracy', 'balanced_accuracy', 'roc_auc', 'tpr_at_1pct_fpr_roc']].round(4))
display(same_year_target_generalization.set_index('target_year').round(4))


,,accuracy,balanced_accuracy,roc_auc,tpr_at_1pct_fpr_roc
target_year,method,,,,
2023,matched_rich_rf,0.5120,0.5120,0.5167,0.0113
2024,matched_rich_rf,0.5098,0.5098,0.5092,0.0104


,train_accuracy,validation_accuracy,train_log_loss,validation_log_loss,validation_minus_train_log_loss,train_minus_validation_accuracy,validation_available,holdout_partition,holdout_accuracy,holdout_log_loss,holdout_minus_train_log_loss,train_minus_holdout_accuracy
target_year,,,,,,,,,,,,
2023,0.8438,0.8235,0.3521,0.3928,0.0407,0.0203,True,test,0.8253,0.3850,0.0329,0.0185
2024,0.8407,0.8203,0.3550,0.3907,0.0357,0.0204,True,test,0.8188,0.3974,0.0424,0.0219


## Per-sample calibration: offline LiRA on the same-year reference pools
Use the already-queried raw binary true-class logit margin, avoiding softmax
rounding/saturation. Each target candidate is absent from every reference's
training **and validation** data. Estimate its OUT location as the median margin
across the 16 reference models. Use a pooled OUT standard deviation fitted only
on attack-training record IDs (floor 1e-6). The membership score is the negative
Gaussian OUT log likelihood, matching the fixed-variance offline LiRA form.
This is a likelihood-based score, not a calibrated probability of membership.

Calibrate cutoffs using shadows as pseudo-targets. For every pseudo-target query,
exclude its own model from the OUT distribution. Also exclude references where
that row was training or validation data: validation can affect early stopping.
Require at least three remaining pure-OUT references; report skipped rows and
the remaining support. Reuse the RF attack's fixed record-group roles. Fit the
accuracy/low-FPR cutoffs on shadow validation only, then freeze them for the
target and every subgroup. No target membership labels fit any distribution,
threshold or variance. Test data never chooses score direction or a method.

**This is offline LiRA, not online LiRA.** Online LiRA would train candidate-IN
and candidate-OUT references and conflict with the requested disjoint data.
The smaller shadow training size, limited reference count, learned-vocabulary
differences and finite calibration support remain limitations. Near-chance
results strengthen evidence about these tested attacks; they do not prove
privacy or establish that no vulnerable subgroup exists.
The 15% OUT-test fraction gives fewer pure-OUT calibration references per
pseudo-target than the former 40% fraction. The minimum remains three; skipped
rows and actual reference support are reported rather than silently relaxed.

References: [LiRA paper](https://arxiv.org/abs/2112.03570),
[reference offline implementation](https://github.com/tensorflow/privacy/blob/master/research/mi_lira_2021/plot.py).


In [13]:
for year, result in same_year_results.items():
    same_year_progress.update(f'{year}: offline LiRA calibration using existing reference queries')
    role_ids = {role: set(frame.record_id) for role, frame in result['rich_attack']['splits'].items()}
    offline = fit_offline_calibration(result['auxiliary_margins'], result['pure_out_masks'],
        result['shadow_rows'], role_ids, result['pool'].index, minimum_out=SAME_YEAR_MINIMUM_OUT)
    rows = result['target_rows']
    assert result['target_reference_margins'].shape == (SAME_YEAR_REFERENCE_COUNT, len(rows))
    rows['offline_out_center'] = np.median(result['target_reference_margins'], axis=0)
    rows['offline_lira_score'] = gaussian_out_score(rows.class_scaled_logit.to_numpy(),
        rows.offline_out_center.to_numpy(), offline['scale'])
    result['offline_lira'] = offline
    metrics = score_metrics(rows.membership, rows.offline_lira_score,
        threshold=offline['accuracy_threshold'], low_fpr_threshold=offline['low_fpr_threshold'])
    same_year_metric_rows.append(dict(target_year=year, auxiliary_year=year, method='offline_lira', **metrics))
    for role, metrics in offline['metrics'].items():
        same_year_aux_rows.append(dict(auxiliary_year=year, method='offline_lira', partition=role, **metrics))
    same_year_slice_parts.append(subgroup_metrics(rows, result['candidates'], rows.outlier_top5,
        score_column='offline_lira_score', threshold=offline['accuracy_threshold'],
        low_fpr_threshold=offline['low_fpr_threshold']).assign(target_year=year, method='offline_lira'))

same_year_mia_metrics = pd.DataFrame(same_year_metric_rows)
same_year_auxiliary_metrics = pd.DataFrame(same_year_aux_rows)
same_year_subgroups = pd.concat(same_year_slice_parts, ignore_index=True)
same_year_progress.update('Same-year RF, offline LiRA and subgroup diagnostics completed')


In [14]:
display(same_year_mia_metrics.set_index(['target_year', 'method'])[
    ['accuracy', 'balanced_accuracy', 'roc_auc', 'tpr_at_1pct_fpr_roc',
     'tpr_at_shadow_threshold', 'fpr_at_shadow_threshold']].round(4))
display(same_year_auxiliary_metrics.set_index(['auxiliary_year', 'method', 'partition'])[
    ['records', 'accuracy', 'balanced_accuracy', 'roc_auc']].round(4))
display(pd.concat([result['attack_balance_audit'].assign(auxiliary_year=year)
    for year, result in same_year_results.items()], ignore_index=True).groupby(
    ['auxiliary_year', 'attack_role'])[['selected_in', 'selected_out']].sum())
display(same_year_subgroups.loc[same_year_subgroups.group_type.eq('outlier')].set_index(
    ['target_year', 'method', 'group'])[['records', 'members', 'nonmembers',
    'majority_class_accuracy', 'accuracy', 'balanced_accuracy', 'roc_auc']].round(4))
display(same_year_subgroups.loc[same_year_subgroups.group_type.isin(['income_gt_50k', 'SEX', 'RAC1P'])].set_index(
    ['target_year', 'method', 'group_type', 'group'])[['records', 'members', 'nonmembers',
    'accuracy', 'balanced_accuracy', 'roc_auc', 'metric_status']].round(4))
display(pd.DataFrame([dict(auxiliary_year=year, reference_count=SAME_YEAR_REFERENCE_COUNT,
    pool_records=len(result['pool']), shadow_train_records=result['model_protocols']['same_year_mlp_01']['partition_sizes']['train'],
    target_train_records=len(target_results[year]['partitions']['train']),
    offline_out_std=result['offline_lira']['scale'], skipped_calibration_rows=result['offline_lira']['skipped_rows'],
    calibration_min_out=result['offline_lira']['out_reference_count_summary']['minimum'],
    calibration_median_out=result['offline_lira']['out_reference_count_summary']['median'],
    calibration_max_out=result['offline_lira']['out_reference_count_summary']['maximum'])
    for year, result in same_year_results.items()]).set_index('auxiliary_year'))


,,accuracy,balanced_accuracy,roc_auc,tpr_at_1pct_fpr_roc,tpr_at_shadow_threshold,fpr_at_shadow_threshold
target_year,method,,,,,,
2023,matched_rich_rf,0.5120,0.5120,0.5167,0.0113,0.0102,0.0090
2024,matched_rich_rf,0.5098,0.5098,0.5092,0.0104,0.0096,0.0095
2023,offline_lira,0.5007,0.5007,0.5025,0.0101,0.0104,0.0112
2024,offline_lira,0.4998,0.4998,0.4994,0.0102,0.0201,0.0195


records  accuracy  \
auxiliary_year method          partition                       
2023           matched_rich_rf train        211234    0.5448   
                               validation    70574    0.5166   
                               test          69936    0.5136   
2024           matched_rich_rf train        211712    0.5312   
                               validation    70738    0.5135   
                               test          70798    0.5120   
2023           offline_lira    train         83857    0.5131   
                               validation    28209    0.5120   
                               test          27225    0.5093   
2024           offline_lira    train         83887    0.5092   
                               validation    28115    0.5118   
                               test          28260    0.5102   

                                           balanced_accuracy  roc_auc  
auxiliary_year method          partition                               
2023           matched_rich_rf train                  0.5448   0.5900  
                               validation             0.5166   0.5206  
                               test                   0.5136   0.5191  
2024           matched_rich_rf train                  0.5312   0.5943  
                               validation             0.5135   0.5155  
                               test                   0.5120   0.5138  
2023           offline_lira    train                  0.5131   0.5184  
                               validation             0.5121   0.5133  
                               test                   0.5093   0.5140  
2024           offline_lira    train                  0.5085   0.5121  
                               validation             0.5100   0.5097  
                               test                   0.5106   0.5110

selected_in  selected_out
auxiliary_year attack_role                           
2023           test               34968         34968
               train             105617        105617
               validation         35287         35287
2024           test               35399         35399
               train             105856        105856
               validation         35369         35369

records  members  \
target_year method          group                                    
2023        matched_rich_rf top_5_percent            1000      494   
                            remaining_95_percent    19000     9506   
2024        matched_rich_rf top_5_percent            1000      518   
                            remaining_95_percent    19000     9482   
2023        offline_lira    top_5_percent            1000      494   
                            remaining_95_percent    19000     9506   
2024        offline_lira    top_5_percent            1000      518   
                            remaining_95_percent    19000     9482   

                                                  nonmembers  \
target_year method          group                              
2023        matched_rich_rf top_5_percent                506   
                            remaining_95_percent        9494   
2024        matched_rich_rf top_5_percent                482   
                            remaining_95_percent        9518   
2023        offline_lira    top_5_percent                506   
                            remaining_95_percent        9494   
2024        offline_lira    top_5_percent                482   
                            remaining_95_percent        9518   

                                                  majority_class_accuracy  \
target_year method          group                                           
2023        matched_rich_rf top_5_percent                          0.5060   
                            remaining_95_percent                   0.5003   
2024        matched_rich_rf top_5_percent                          0.5180   
                            remaining_95_percent                   0.5009   
2023        offline_lira    top_5_percent                          0.5060   
                            remaining_95_percent                   0.5003   
2024        offline_lira    top_5_percent                          0.5180   
                            remaining_95_percent                   0.5009   

                                                  accuracy  balanced_accuracy  \
target_year method          group                                               
2023        matched_rich_rf top_5_percent           0.5120             0.5154   
                            remaining_95_percent    0.5119             0.5118   
2024        matched_rich_rf top_5_percent           0.5220             0.5104   
                            remaining_95_percent    0.5092             0.5098   
2023        offline_lira    top_5_percent           0.4970             0.4978   
                            remaining_95_percent    0.5009             0.5009   
2024        offline_lira    top_5_percent           0.5150             0.5188   
                            remaining_95_percent    0.4990             0.4989   

                                                  roc_auc  
target_year method          group                          
2023        matched_rich_rf top_5_percent          0.5196  
                            remaining_95_percent   0.5166  
2024        matched_rich_rf top_5_percent          0.5085  
                            remaining_95_percent   0.5094  
2023        offline_lira    top_5_percent          0.5016  
                            remaining_95_percent   0.5026  
2024        offline_lira    top_5_percent          0.5280  
                            remaining_95_percent   0.4980

records  members  nonmembers  \
target_year method          group_type    group                                 
2023        matched_rich_rf income_gt_50k 0        10032     5016        5016   
                                          1         9968     4984        4984   
                            SEX           1.0      10637     5306        5331   
                                          2.0       9363     4694        4669   
                            RAC1P         1.0       8271     4065        4206   
                                          2.0        803      414         389   
                                          3.0        297      139         158   
                                          4.0          3        2           1   
                                          5.0         61       33          28   
                                          6.0       3653     1878        1775   
                                          7.0         56       29          27   
                                          8.0       3417     1736        1681   
                                          9.0       3439     1704        1735   
2024        matched_rich_rf income_gt_50k 0         9584     4792        4792   
                                          1        10416     5208        5208   
                            SEX           1.0      10590     5317        5273   
                                          2.0       9410     4683        4727   
                            RAC1P         1.0       8159     4078        4081   
                                          2.0        828      401         427   
                                          3.0        280      148         132   
                                          4.0          3        1           2   
                                          5.0         61       33          28   
                                          6.0       3720     1851        1869   
                                          7.0         84       49          35   
                                          8.0       3370     1693        1677   
                                          9.0       3495     1746        1749   
2023        offline_lira    income_gt_50k 0        10032     5016        5016   
                                          1         9968     4984        4984   
                            SEX           1.0      10637     5306        5331   
                                          2.0       9363     4694        4669   
                            RAC1P         1.0       8271     4065        4206   
                                          2.0        803      414         389   
                                          3.0        297      139         158   
                                          4.0          3        2           1   
                                          5.0         61       33          28   
                                          6.0       3653     1878        1775   
                                          7.0         56       29          27   
                                          8.0       3417     1736        1681   
                                          9.0       3439     1704        1735   
2024        offline_lira    income_gt_50k 0         9584     4792        4792   
                                          1        10416     5208        5208   
                            SEX           1.0      10590     5317        5273   
                                          2.0       9410     4683        4727   
                            RAC1P         1.0       8159     4078        4081   
                                          2.0        828      401         427   
                                          3.0        280      148         132   
                                          4.0          3        1           2   
                                          5.0         61       33          28   

,reference_count,pool_records,shadow_train_records,target_train_records,offline_out_std,skipped_calibration_rows,calibration_min_out,calibration_median_out,calibration_max_out
auxiliary_year,,,,,,,,,
2023,16,73278,54958,100000,1.983934,212453,3,3.0,10
2024,16,73591,55192,100000,1.910457,212986,3,3.0,10


## Export the additional same-year results
Save these results under the split-specific `same_year_matched_mlp_train75_val10_test15`
(or `same_year_matched_mlp_train80_test20`) directory, retaining the
existing cross-year baseline files. Save metrics, subgroup counts, disjointness
audits, reference recipes/epoch diagnostics, histories and final target scores.
No reference models, fitted anomaly models, checkpoints or log files are saved.
The reference score matrices remain in memory for this run only.

For the report, describe this as an architecture-informed, same-year attack
comparison with matched MLP hyperparameters and preprocessing recipe but smaller
reference training sets. Report adverse subgroup results as well as near-chance
results; do not select or suppress groups after inspecting scores. Whole-set
chance is 50% because target MIA candidates are balanced. Slice baselines depend
on their actual membership proportions. One target seed per year and these
attacks cannot certify privacy or rule out untested attacks.


In [15]:
def same_year_json_value(value):
    if isinstance(value, dict):
        return {str(key): same_year_json_value(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [same_year_json_value(item) for item in value]
    if isinstance(value, np.generic):
        return same_year_json_value(value.item())
    if isinstance(value, float) and not np.isfinite(value):
        return None if np.isnan(value) else str(value)
    return value

SAME_YEAR_DIR.mkdir(parents=True, exist_ok=True)
same_year_mia_metrics.to_csv(SAME_YEAR_DIR / 'same_year_mia_metrics.csv', index=False)
same_year_auxiliary_metrics.to_csv(SAME_YEAR_DIR / 'same_year_auxiliary_attack_metrics.csv', index=False)
same_year_subgroups.to_csv(SAME_YEAR_DIR / 'same_year_subgroup_metrics.csv', index=False)
same_year_target_generalization.to_csv(SAME_YEAR_DIR / 'target_generalization.csv', index=False)
same_year_shadow_generalization.to_csv(SAME_YEAR_DIR / 'shadow_generalization.csv', index=False)
same_year_disjointness.to_csv(SAME_YEAR_DIR / 'disjointness_audit.csv', index=False)
for year, result in same_year_results.items():
    directory = SAME_YEAR_DIR / str(year)
    directory.mkdir(parents=True, exist_ok=True)
    pd.Series(result['pool'].index, name='record_id').to_csv(directory / 'shadow_pool_ids.csv', index=False)
    result['attack_balance_audit'].to_csv(directory / 'attack_balance_audit.csv', index=False)
    pd.DataFrame([dict(record_id=record_id, attack_role=role)
        for role, ids in result['attack_role_ids'].items() for record_id in sorted(ids)]).to_csv(
            directory / 'attack_record_roles.csv', index=False)
    result['target_rows'].to_csv(directory / 'target_mia_scores.csv')
    for name, history in result['histories'].items():
        pd.DataFrame(history).to_csv(directory / f'{name}_validation_history.csv', index=False)
same_year_protocol = dict(execution_status='same_year_attacks_completed',
    pairs=[dict(target_year=year, auxiliary_year=year) for year in SAME_YEAR_TARGETS],
    references_per_year=SAME_YEAR_REFERENCE_COUNT, shadow_algorithm='MLP only',
    matched_target_training_recipe=TRAIN_CONFIG, independent_reference_seeds=True,
    preprocessing='Same common47 recipe, fitted on each model training partition only; vocabulary may differ',
    shadow_data='Unused same-year rows excluding every target training/validation/test/MIA ID',
    shadow_split=('75% train, 10% validation, 15% test' if TRAIN_CONFIG['early_stopping']
        else '80% train, 20% test; no validation or best-epoch restoration'),
    shadow_split_rounding='Ceil test and validation counts; remainder to training; independent seeds',
    standard_attack_sampling='All OUT-test rows; exact matching IN-training subsample within each fixed record role',
    attack_roles_before_subsampling=True, standard_attack_exact_balance_per_role=True,
    training_size_matched=False, target_retrained=False,
    attack_features=next(iter(same_year_results.values()))['features'],
    rf_attack='300 trees, max_depth12, min_samples_leaf20, sqrt features; grouped 60/20/20 roles',
    thresholds='Shadow validation only; frozen for target and subgroup evaluation',
    blind_walk=INCLUDE_BLIND_WALK, walk_config=WALK_CONFIG, numeric_protocol=EXPANDED_NUMERIC_WALK,
    offline_lira=dict(method='Negative OUT Gaussian log likelihood, per-record median, fixed pooled OUT std',
        variance_fit='Attack-training IDs only; no target scores', minimum_out=SAME_YEAR_MINIMUM_OUT,
        variance_floor=1e-6, reference_query_exclusion=True, exclude_shadow_validation_from_out=True,
        online_in_models=False, score_direction='Higher negative OUT log likelihood means member',
        implementation_reference='https://github.com/tensorflow/privacy/blob/master/research/mi_lira_2021/plot.py'),
    outliers=dict(method='IsolationForest', fit_data='Disjoint same-year shadow pool only',
        n_estimators=100, max_samples=256, seed=SEED, top_fraction=.05,
        ranking='Target inputs only, stable tie handling; membership labels not used'),
    subgroup_fields=['income_gt_50k', 'SEX', 'RAC1P'], loss_gap='validation log loss minus training log loss',
    years={str(year): dict(pool_records=len(result['pool']), model_protocols=result['model_protocols'],
        offline_calibration_skipped_rows=result['offline_lira']['skipped_rows'],
        offline_calibration_kept_rows=result['offline_lira']['kept_rows'],
        offline_out_reference_count_summary=result['offline_lira']['out_reference_count_summary'],
        offline_scale=result['offline_lira']['scale'], target_generalization=result['target_generalization'],
        rich_rf_threshold=result['rich_attack']['accuracy_threshold'],
        rich_rf_low_fpr_threshold=result['rich_attack']['low_fpr_threshold'],
        offline_threshold=result['offline_lira']['accuracy_threshold'],
        offline_low_fpr_threshold=result['offline_lira']['low_fpr_threshold'])
        for year, result in same_year_results.items()},
    checkpoints_enabled=False, logging_enabled=False,
    limitations=['Smaller reference training sets than target', 'Finite OUT reference support',
        'Learned encoder vocabularies/input dimensions may differ', 'Offline rather than online LiRA',
        'One target seed per year', 'Near-chance scores do not prove privacy'])
_ = (SAME_YEAR_DIR / 'protocol.json').write_text(
    json.dumps(same_year_json_value(same_year_protocol), indent=2, allow_nan=False), encoding='utf-8')
print('Additional same-year results:', SAME_YEAR_DIR)


Additional same-year results: artifacts\exp_1_generalization_targets2023_2024_aux2014_2018_s16_common47_mixed_mlp\with_walk_consistent_v1\same_year_matched_mlp_train75_val10_test15
